# KaBuM — Scraper Multi-Categoria
Coleta dados de CPU, GPU, Placa-mãe, SSD, Fonte e RAM usando `__NEXT_DATA__`.

In [1]:
import requests
import json
import time
import random
import pathlib
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date

In [2]:
CATEGORIAS = {
    "cpu":       "https://www.kabum.com.br/hardware/processadores",
    "gpu":       "https://www.kabum.com.br/hardware/placa-de-video-vga",
    "placa_mae": "https://www.kabum.com.br/hardware/placas-mae",
    "ssd":       "https://www.kabum.com.br/hardware/ssd-2-5",
    "fonte":     "https://www.kabum.com.br/hardware/fontes",
    "ram":       "https://www.kabum.com.br/hardware/memoria-ram",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "pt-BR,pt;q=0.9",
}

DATA_COLETA = date.today().isoformat()
NUM_PAGINAS = 20

In [3]:
def extrair_produtos(html):
    """Extrai a lista de produtos do __NEXT_DATA__ da página."""
    soup = BeautifulSoup(html, "html.parser")
    script_next = soup.find("script", id="__NEXT_DATA__")
    catalog = json.loads(json.loads(script_next.string)["props"]["pageProps"]["data"])
    return catalog["catalogServer"]["data"]


def parsear_produto(produto, categoria_key):
    """Mapeia os campos do produto para o formato do DataFrame."""
    return {
        "data_coleta":     DATA_COLETA,
        "categoria_key":   categoria_key,
        "id":              produto.get("code"),
        "nome":            produto.get("name"),
        "preco_original":  produto.get("oldPrice"),
        "preco_atual":     produto.get("price"),
        "preco_pix":       produto.get("priceWithDiscount"),
        "desconto_pct":    produto.get("discountPercentage"),
        "avaliacao":       produto.get("rating"),
        "num_avaliacoes":  produto.get("ratingCount"),
        "disponivel":      produto.get("available"),
        "fabricante":      produto.get("manufacturer", {}).get("name"),
        "categoria":       produto.get("category"),
        "garantia":        produto.get("warranty"),
        "frete_gratis":    produto.get("flags", {}).get("isFreeShipping"),
        "url":             f"https://www.kabum.com.br/produto/{produto.get('code')}/{produto.get('friendlyName', '')}",
    }


def scrape_categoria(nome, url, num_paginas=NUM_PAGINAS):
    """Raspa todas as páginas de uma categoria."""
    todos = []
    print(f"\n{'='*50}")
    print(f"Coletando: {nome.upper()} — {url}")

    for pagina in range(1, num_paginas + 1):
        url_pag = f"{url}?page_number={pagina}"
        try:
            resp = requests.get(url_pag, headers=HEADERS, timeout=15)
            resp.raise_for_status()
            produtos_raw = extrair_produtos(resp.text)
            if not produtos_raw:
                print(f"  Página {pagina}: sem produtos — encerrando")
                break
            todos.extend([parsear_produto(p, nome) for p in produtos_raw])
            print(f"  Página {pagina}: {len(produtos_raw)} produtos (total: {len(todos)})")
        except Exception as e:
            print(f"  Página {pagina}: ERRO — {e}")
            break
        time.sleep(random.uniform(0.5, 1.5))

    print(f"  Total final: {len(todos)} produtos")
    return todos

In [4]:
import os

# 00_Dados/ mora na raiz do repositorio (um nivel acima deste notebook).
# Defina a variavel de ambiente KABUM_DATA_ROOT para apontar para outro lugar
# sem editar o codigo.
data_dir = pathlib.Path(os.environ.get("KABUM_DATA_ROOT", str(pathlib.Path("..", "00_Dados")))) / DATA_COLETA
data_dir.mkdir(exist_ok=True)

resultados = {}

for nome, url in CATEGORIAS.items():
    produtos = scrape_categoria(nome, url)
    df = pd.DataFrame(produtos)
    resultados[nome] = df
    arquivo = data_dir / f"kabum_{nome}_{DATA_COLETA}.csv"
    df.to_csv(arquivo, index=False, encoding="utf-8-sig")
    print(f"  Salvo: {arquivo}")

print("\n✓ Coleta concluída!")


Coletando: CPU — https://www.kabum.com.br/hardware/processadores


  Página 1: 60 produtos (total: 60)


  Página 2: 60 produtos (total: 120)


  Página 3: 60 produtos (total: 180)


  Página 4: 60 produtos (total: 240)


...

  Página 17: 60 produtos (total: 1020)


  Página 18: 60 produtos (total: 1080)


  Página 19: 60 produtos (total: 1140)


  Página 20: 60 produtos (total: 1200)


  Total final: 1200 produtos
  Salvo: ..\00_Dados\2026-08-11\kabum_ram_2026-08-11.csv

✓ Coleta concluída!


In [5]:
# Consolida tudo num único CSV
df_consolidado = pd.concat(resultados.values(), ignore_index=True)
arquivo_consolidado = data_dir / f"kabum_todas_pecas_{DATA_COLETA}.csv"
df_consolidado.to_csv(arquivo_consolidado, index=False, encoding="utf-8-sig")

print(f"CSV consolidado: {arquivo_consolidado}")
print(f"Total de produtos: {len(df_consolidado)}\n")
print(df_consolidado.groupby("categoria_key").size().rename("qtd_produtos"))

CSV consolidado: ..\00_Dados\2026-08-11\kabum_todas_pecas_2026-08-11.csv
Total de produtos: 4797

categoria_key
cpu           723
fonte         718
gpu           479
placa_mae     819
ram          1200
ssd           858
Name: qtd_produtos, dtype: int64


In [6]:
# Verificação rápida — 3 primeiros de cada categoria
colunas = ["nome", "preco_pix", "fabricante", "disponivel"]
for nome, df in resultados.items():
    print(f"\n--- {nome.upper()} ---")
    display(df[colunas].head(3))


--- CPU ---


,nome,preco_pix,fabricante,disponivel
0,"Processador AMD Ryzen 7 5700X, 3.4GHz (4.6GHz ...",2163.95,AMD,True
1,"Processador AMD Ryzen 7 7800X3D, 4.2GHz (5.0GH...",3599.99,AMD,True
2,"Processador AMD Ryzen 5 5500, 3.6GHz (4.2GHz M...",1150.44,AMD,True



--- GPU ---


,nome,preco_pix,fabricante,disponivel
0,Placa de Vídeo XFX Swift RX 9070 XT TRIPLE FAN...,5599.99,XFX,True
1,Placa de Vídeo ASRock RX 9060 XT CL 16GO AMD R...,3999.99,ASRock,True
2,Placa De Vídeo MSI RTX 3050 LP OC NVIDIA Gefor...,1699.90,MSI,True



--- PLACA_MAE ---


...

,nome,preco_pix,fabricante,disponivel
0,"SSD Rise Mode Gamer Line, 120GB, SATA III, Lei...",159.99,Rise Mode,True
1,"SSD Kingston NV3, 1 TB, M.2 2280, PCIe 4.0 x4,...",1084.99,Kingston,True
2,"SSD Rise Mode Gamer Line, 240GB, SATA III, Lei...",279.99,Rise Mode,True



--- FONTE ---


,nome,preco_pix,fabricante,disponivel
0,"Fonte Gigabyte UD850GM PG5 V2, 850W, 80 PLUS G...",699.99,Gigabyte,True
1,"Fonte Gigabyte UD850GM, 850W, 80 PLUS Gold, Mo...",699.99,Gigabyte,True
2,"Fonte MSI MAG A650BN, 650W, 80 Plus Bronze, PF...",349.99,MSI,True



--- RAM ---


,nome,preco_pix,fabricante,disponivel
0,"Memória RAM Kingston Fury Beast, 16GB, 3200MHz...",1099.99,Kingston,True
1,"Memória RAM Husky Impulse, 8GB, 3200MHz, DDR4,...",629.99,Husky,True
2,"Memória RAM Rise Mode Z, 8GB, 3200MHz, DDR4, C...",387.99,Rise Mode,True
